# Lab: Preprocess Data

**BINF 6210/8210 - Machine Learning for Bioinformatics**

Learning goals:
- audit structure and quality
- examine missingness
- encode and scale features;
- understand data leakage
- produce Machine Learning-Ready Data from raw data table


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

## Random number generator

https://numpy.org/devdocs/reference/random/generator.html

In [ ]:
# random number generator
RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

# rng.random() # produce random numbers in [0.0, 1.0) that follow a uniform distribution
# rng.integer() # produce random integers in a specified range
# rng.standard_normal()
# rng.normal()

## Simulate a dataset

- sample_id: ['S000', 'S001', 'S002', ...]
- age: normally distributed at loc=52, and scale=14, clipped to between a specified range
- bmi: normally distributed at loc=27, scale=5, clipped to between a specified range
- batch: three batches ["B1", "B2", "B3"] with specified proportion
- sex: Female and Male at 50% percentage each
- metabolite_a: follow a lognormal distribution, i.e. the natural logarithm follows a normal distribution with the specified location and scale
- metabolite_b: follow a lognormal distribution, i.e. the natural logarithm follows a normal distribution with the specified location and scale



- log-odds score for each sample:
$$
    \text{logit} = -3 + 0.045 \times \text{age} + 0.7 \times \ln (1 + \text{metabolite}_a) + 0.35 \times I(\text{sex} = \text{Male})
$$
where:
    - logit represent the log-odds of having or developing a disease
        $$
    \text{logit} = \ln \frac{p}{1-p}
        $$
    where $p = P(\text{disease} = 1 \mid \text{age}, \text{metabolite\_a}, \text{sex})$
    - -3: baseline intercept
    - 0.045 $\times$ age: older age increases the log-odds
    - $\ln (1 + \text{metabolite}_a)$ reduces the skewness and influence of very large metabolite values.
    - 0.7: sets the strength of the metabolite's effect
    - $\mathbf{I}(\text{sex})$ is an indicator function:
        $$
\mathbf{I}(\text{sex} = \text{Male}) =
\begin{cases}
1, & \text{if sex is Male}, \\
0, & \text{otherwise}.
\end{cases}
        $$

    - Example: For one person who is 50 years old, male, and has metabolite_a = 4:
        $$
        \text{logit} = -3 + 0.045 (50) + 0.7 \ln(1+4) + 0.35 \approx 0.727
        $$


- Convert the log-odds to a probability using the logistic function:
$$
\text{probability} = \frac{1}{1+e^{-\text{logit}}}
$$


In [3]:
# simulate a dataset
n = 240 # number of samples

# simulating a dataset with random numbers

df = pd.DataFrame({
    "sample_id": [f"S{i:03d}" for i in range(n)], # sample identifier
    "age": rng.normal(52, 14, n).clip(18, 85),
    "bmi": rng.normal(27, 5, n).clip(16, 48),
    "batch": rng.choice(["B1", "B2", "B3"], n, p=[.45,.35,.20]),
    "sex": rng.choice(["Female", "Male"], n),
    "metabolite_a": rng.lognormal(1.2, .8, n),
    "metabolite_b": rng.lognormal(.5, .6, n),
})

# calculate the logit using the simulated feature values
logit = -3 + .045*df.age + .7*np.log1p(df.metabolite_a) + .35*(df.sex=="Male")

# convert the logit to probability
case_probability = 1/(1+np.exp(-logit))
df["case"] = rng.binomial(n=1, p=case_probability)

# insert missing values into three columns at different rates to create a more realistic dataset with missing data
for col, frac in {"bmi":.08, "metabolite_a":.12, "batch":.04}.items():
    df.loc[rng.choice(n, int(n*frac), replace=False), col] = np.nan

# create a very large outlier in bmi
df.loc[5, "bmi"] = 120  # deliberate data-quality issue

df.head(10)


,sample_id,age,bmi,batch,sex,metabolite_a,metabolite_b,case
0,S000,67.857657,27.054928,B1,Male,0.868678,2.052980,1
1,S001,69.703557,31.394770,B1,Female,1.512129,4.218450,1
2,S002,32.095309,19.513134,B2,Male,4.141804,2.237134,0
3,S003,47.627024,20.448516,B3,Female,7.362735,0.568901,1
4,S004,71.013695,16.811357,B3,Female,2.992805,2.164950,1
5,S005,59.113107,120.000000,B2,Female,3.511666,3.454640,1
6,S006,54.733682,21.408471,B2,Female,5.243238,1.443459,1
7,S007,48.952414,NaN,B1,Male,1.526084,2.996308,0
8,S008,46.626787,30.306891,B2,Female,5.999967,1.089731,0
9,S009,56.035067,23.792286,B2,Male,6.394732,0.676161,0


## 1. Audit before changing anything

Ask: What is one row? What is the prediction target? Which columns are identifiers, features, outcomes?


In [6]:
print('Data shape:')
print(df.shape)

# get data type of each column
print('\nData types:')
display(df.dtypes.to_frame("dtype"))

# get the fraction of data that is missing and sort by extent of missingness
print('\nMissingness:')
display(df.isna().mean().sort_values(ascending=False).to_frame("missing_fraction"))

# generate descriptive statistics for all columns of the data
print('\nDescriptive statistics:')
display(df.describe(include="all").T)


Data shape:
(240, 8)

Data types:


,dtype
sample_id,str
age,float64
bmi,float64
batch,str
sex,str
metabolite_a,float64
metabolite_b,float64
case,int64



Missingness:


,missing_fraction
metabolite_a,0.116667
bmi,0.079167
batch,0.037500
sample_id,0.000000
age,0.000000
sex,0.000000
metabolite_b,0.000000
case,0.000000



Descriptive statistics:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
sample_id,240,240,S000,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,240.0,NaN,NaN,NaN,50.802157,13.898425,18.0,42.274803,51.506228,60.088043,85.0
bmi,221.0,NaN,NaN,NaN,26.509958,7.938253,16.0,22.64392,26.529027,29.489615,120.0
batch,231,3,B1,112,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sex,240,2,Male,123,NaN,NaN,NaN,NaN,NaN,NaN,NaN
metabolite_a,212.0,NaN,NaN,NaN,4.833423,4.145549,0.574627,2.120842,3.431587,6.165053,26.652982
metabolite_b,240.0,NaN,NaN,NaN,1.976731,1.319503,0.296144,1.107255,1.656843,2.564572,10.336887
case,240.0,NaN,NaN,NaN,0.6375,0.481727,0.0,0.0,1.0,1.0,1.0


### Write down issues you have observed.
Identify at least four concerns. For each, write whether it is a **valid value**, **measurement issue**, **missingness issue**, or **modeling decision**.


In [5]:
# Your notes here
data_quality_notes = []
data_quality_notes


[]

## 2. Missing values are data, not merely empty cells

Median/mode imputation is a defensible baseline, but it does not recover the unknown truth. The strategy must match the scientific mechanism and prediction setting.


In [7]:
df.groupby("case")[["bmi", "metabolite_a"]].agg(["count", "median", "mean"])

bmi                       metabolite_a                    
     count     median       mean        count    median      mean
case                                                             
0       79  26.019979  25.747941           81  3.044733  4.410657
1      142  26.535664  26.933898          131  3.910891  5.094828

## 3. Split before learning preprocessing parameters

Split data into training and testing. Training data is for training a model and testing data is for assessing the performance of the model.


In [8]:
X = df.drop(columns=["case","sample_id"])
y = df["case"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.25, stratify=y, random_state=RANDOM_STATE)
print(X_train.shape, X_test.shape, y_train.mean(), y_test.mean())


(180, 6) (60, 6) 0.6388888888888888 0.6333333333333333


## 4. Impute missing values and encode categorical variables


In [10]:
# specify variables in the continuous and categorical categories
numeric_variables = ["age", "bmi", "metabolite_a", "metabolite_b"]
categorical_variables = ["batch", "sex"]

# Continuous data pipeline
# Numerical columns go through two steps: missing value imputation and standardization
# Missing values in each numerical column are replaced with that column's median calculated from X_train
# The names "impute" and "scale" are labels that make the individual steps accessible later.
numerical_data_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

# Categorical data pipeline
# Categorical data go through two steps: missing value imputation and one-hot encoding
# Missing values are replaced with the most common category in each training column.
# Each category is converted into a binary indicator column. handle_unknown="ignore" prevents an error if future data contain a category that was not present in X_train. That unseen category will be represented by zeros in all the corresponding indicator columns.
categorical_data_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore"))
])

# Combine the two pipelines
# ColumnTransformer applies numerical_data_pipeline to the columns in numerical_variables and applies categorical_data_pipeline to the columns in categorical_variables. Finally, it joins all the processed columns into one feature matrix. The names "numerical" and "categorical" are names for the two transformer branches.
preprocess = ColumnTransformer([
    ("numerical", numerical_data_pipeline, numeric_variables),
    ("categorical", categorical_data_pipeline, categorical_variables)])

# Fit and transform the training data
# fit: leans the medians, means, standard deviations, most frequent categories, and one-hot categories from X_train.
# transform: applies those learned preprocessing rules to X_train
X_train_transformed = preprocess.fit_transform(X_train)

print("Transformed training shape:", X_train_transformed.shape)
print(preprocess.get_feature_names_out())

Transformed training shape: (180, 9)
['numerical__age' 'numerical__bmi' 'numerical__metabolite_a'
 'numerical__metabolite_b' 'categorical__batch_B1' 'categorical__batch_B2'
 'categorical__batch_B3' 'categorical__sex_Female' 'categorical__sex_Male']


## 5. Data leakage

You should not call fit_transform() on the test data because that would learn information from the test data and cause data leakage. Data leakage occurs when information that would not legitimately be available when making a prediction influences model training. It can make a model appear much more accurate during evaluation than it will be on genuinely new data.

In [ ]:
# For validation or test data, use only transform()

X_test_transformed = preprocess.transform(X_test)

## 5. Exit ticket

1. Which objects learned parameters from the training data?
2. Why is `sample_id` excluded?
3. What would happen if a new batch label appeared in the test set?
